# Assignment 2: PeopleNet (person / non-person), transfer learning

**Approach**
1. Pre-trained backbone from the template's list: `efficientnet_b3_ra2_imagenet` (ImageNet).
2. Model: `raw pixels → ImageConverter (resize + ImageNet normalization) → EfficientNet-B3 →
   average + max pooling → dropout → Dense(1, sigmoid)`.
3. **Stage 1:** backbone frozen, train only the new head (1 epoch).
4. **Stage 2:** fine-tune the whole backbone (BatchNorm layers stay frozen) with a low, cosine-decayed
   learning rate and light augmentation (flip, small zoom, brightness).
5. The best epoch by **validation accuracy at 0.5** (the graded metric) is kept, rebuilt as a plain float32
   model, and used to score the 20K score images.

**Fixes over the template:** the template feeds raw 0..255 pixels into the backbone (no normalization), and its
`Dense(1)` has no sigmoid (so `binary_crossentropy` gets logits: loss 8.0, accuracy 0.50, outputs outside [0, 1]).

**Instance:** `ml.g4dn.4xlarge` (T4 GPU, 64 GB RAM). Expected total run time: roughly 35-45 minutes.
Stop the space when you are done.

**If the kernel dies during Stage 2:** the best weights so far are saved after every epoch to `pnp_final/best.weights.h5`.
Re-run the cells up to and including "Model" and the data-pipeline cell, skip the training cells, and run from "Final model" on.

In [ ]:
import os
# Keras backend: must be set before keras is imported.
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

PRESET = 'efficientnet_b3_ra2_imagenet'   # 'efficientnet_b0_ra_imagenet' is ~3x cheaper and ~1-2 points less accurate
BATCH_SIZE = 32
STAGE1_EPOCHS = 1        # head only
FINETUNE_EPOCHS = 4      # whole backbone; each epoch is roughly 6-8 minutes on a T4 with B3
FINETUNE_LR = 1e-4
VAL_FRACTION = 0.1       # 8K labeled images held out to pick the best epoch
SEED = 11
LIMIT = None             # only for a quick code check on a CPU instance (e.g. 512); None = full dataset

WORK_DIR = 'pnp_final'           # persistent home volume: checkpoint and small files
CACHE_DIR = '/tmp/pnp_cache'     # decoded pixels (~15 GB); lost when the space stops, that's fine
MODEL_DIR = 'pnp_model'          # the submission files
for d in [WORK_DIR, CACHE_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

In [ ]:
!ls pnp_dataset.zip || (aws s3 cp s3://danylo-ucla/pnp_dataset.zip ./)

In [ ]:
import keras
import tensorflow as tf
import json, time, shutil, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import PIL.Image
import keras_hub
from concurrent.futures import ThreadPoolExecutor

# Harmless warnings: a finite tf.data pipeline ending, and input-name structure in keras_hub models.
warnings.filterwarnings('ignore', message='.*ran out of data.*')
warnings.filterwarnings('ignore', message='.*structure of `inputs`.*')

print("Keras", keras.__version__, "| keras_hub", keras_hub.__version__, "| TF", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    print(gpu)
    tf.config.experimental.set_memory_growth(gpu, True)

# Fail early with a clear message instead of crashing an hour in.
ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
print(f'RAM: {ram_gb:.0f} GB')
if LIMIT is None:
    assert gpus, 'No GPU found: switch the space to ml.g4dn.4xlarge for the full run.'
    assert ram_gb > 30, 'Not enough RAM for the decoded dataset (~15 GB): use ml.g4dn.4xlarge.'

## Data
Images are decoded once from the zip (parallel threads) into `uint8` arrays in RAM, and cached in `/tmp`
so that a kernel restart does not need to decode again. The order is the same as the template's
`build_dataset` (sorted file names), so labels and score rows line up.

In [ ]:
def list_images(z: zipfile.ZipFile, segment: str) -> list:
    # Same selection and order as collegium.m02_cnn.utils.pnp_dataset.load_images
    return sorted(p for p in z.namelist()
                  if p.startswith(f'pnp_dataset/{segment}') and 'npy' not in p and not p.endswith('/'))


def _decode_chunk(names, out, start):
    with zipfile.ZipFile('pnp_dataset.zip') as z:
        for k, name in enumerate(names):
            with z.open(name) as f:
                img = np.asarray(PIL.Image.open(f).convert('RGB'), dtype=np.uint8)
            assert img.shape == (224, 224, 3), (name, img.shape)
            out[start + k] = img


def decode_segment(segment: str, limit=None) -> np.ndarray:
    cache = f'{CACHE_DIR}/{segment}_x_{limit or "all"}.npy'
    if os.path.exists(cache):
        print(f'{segment}: loading cached pixels from {cache}')
        return np.load(cache)

    with zipfile.ZipFile('pnp_dataset.zip') as z:
        names = list_images(z, segment)
    if limit is not None:
        names = names[:limit]

    out = np.empty((len(names), 224, 224, 3), dtype=np.uint8)
    chunk = 500
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=os.cpu_count()) as ex:
        list(ex.map(lambda i: _decode_chunk(names[i:i + chunk], out, i), range(0, len(names), chunk)))
    print(f'{segment}: decoded {len(names)} images in {time.time() - t0:.0f}s ({out.nbytes / 1e9:.1f} GB)')

    if shutil.disk_usage(CACHE_DIR).free > out.nbytes * 1.2:
        np.save(cache, out)
    return out


train_x = decode_segment('train', LIMIT)
score_x = decode_segment('score', None if LIMIT is None else min(LIMIT, 1000))

with zipfile.ZipFile('pnp_dataset.zip') as z:
    with z.open('pnp_dataset/train_y.npy') as f:
        train_y = np.load(f).astype(np.int32)
train_y = train_y[:len(train_x)]

print('train', train_x.shape, 'score', score_x.shape)
print('person share in train:', train_y.mean().round(4))
if LIMIT is None:
    assert len(train_x) == len(train_y) == 80000 and len(score_x) == 20000

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified split: validation has the same person / non-person mix as training.
tr_idx, va_idx = train_test_split(np.arange(len(train_x)), test_size=VAL_FRACTION,
                                  stratify=train_y, random_state=SEED)
tr_idx, va_idx = np.sort(tr_idx), np.sort(va_idx)
print(f'train: {len(tr_idx)} images ({train_y[tr_idx].mean():.3f} person) | '
      f'val: {len(va_idx)} images ({train_y[va_idx].mean():.3f} person)')

In [ ]:
# A look at the data: 8 person and 8 non-person images.
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for row, label in enumerate([1, 0]):
    for ax, i in zip(axes[row], np.flatnonzero(train_y == label)[:8]):
        ax.imshow(train_x[i]); ax.axis('off')
    axes[row][0].set_title('person' if label else 'non-person', loc='left')
plt.tight_layout(); plt.show()

## Model
* `ImageConverter` applies the backbone's own preprocessing (ImageNet mean/std normalization, resize to the
  resolution it was trained at: 288px for B3). Bilinear resizing is used because the GPU's XLA compiler
  does not support the preset's default bicubic resize.
* Average pooling summarizes the whole image; max pooling keeps the strongest local response, which helps
  with small people in large scenes.
* Only built-in Keras and keras_hub layers are used, so the saved model loads without any custom code.

In [ ]:
def build_model(preset: str = PRESET, load_weights: bool = True):
    native = tuple(keras_hub.layers.ImageConverter.from_preset(preset).get_config()['image_size'])
    if native == (224, 224):
        converter = keras_hub.layers.ImageConverter.from_preset(preset, image_size=None, name='preprocess')
    else:
        converter = keras_hub.layers.ImageConverter.from_preset(
            preset, interpolation='bilinear', antialias=False, name='preprocess')
    backbone = keras_hub.models.Backbone.from_preset(preset, load_weights=load_weights)

    pixels = keras.Input((224, 224, 3), name='pixels')     # raw RGB values 0..255, like the template's dataset
    fmap = backbone(converter(pixels))
    features = keras.layers.Concatenate(name='features')([
        keras.layers.GlobalAveragePooling2D(name='avg_pool')(fmap),
        keras.layers.GlobalMaxPooling2D(name='max_pool')(fmap),
    ])
    features = keras.layers.Dropout(0.3, name='dropout')(features)
    # Zero-initialized output layer: training starts from p = 0.5 everywhere (no overconfident random head).
    prob = keras.layers.Dense(1, activation='sigmoid', kernel_initializer='zeros', dtype='float32',
                              name='person_prob')(features)
    return keras.Model(pixels, prob, name='pnp_model'), backbone, converter


def set_backbone_trainable(backbone, trainable: bool):
    backbone.trainable = trainable
    if trainable:
        # Keep BatchNorm frozen (inference mode, ImageNet statistics): standard for fine-tuning.
        for layer in backbone._flatten_layers(include_self=False, recursive=True):
            if isinstance(layer, keras.layers.BatchNormalization):
                layer.trainable = False


def compile_model(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[keras.metrics.BinaryAccuracy(name='acc', threshold=0.5)],
        jit_compile=False,   # plain cuDNN kernels: no XLA surprises
    )


# Training runs in mixed precision (float16 on the T4 tensor cores, float32 weights): about 2x faster.
keras.backend.clear_session()
keras.utils.set_random_seed(SEED)
keras.config.set_dtype_policy('mixed_float16' if gpus else 'float32')
model, backbone, converter = build_model()
keras.config.set_dtype_policy('float32')

print('ImageConverter:', {k: converter.get_config()[k] for k in ['image_size', 'interpolation', 'scale', 'offset']})
model.summary(line_length=100)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def augment(x, y):
    # x: float32 batch of raw pixels (B, 224, 224, 3). Light augmentation only: a person must stay in the image.
    b = tf.shape(x)[0]
    x = tf.image.random_flip_left_right(x)
    scale = tf.random.uniform([b, 1], 0.85, 1.0)                  # zoom in by up to 15%
    oy = tf.random.uniform([b, 1]) * (1.0 - scale)
    ox = tf.random.uniform([b, 1]) * (1.0 - scale)
    boxes = tf.concat([oy, ox, oy + scale, ox + scale], axis=1)
    x = tf.image.crop_and_resize(x, boxes, tf.range(b), (224, 224))
    x = x * tf.random.uniform([b, 1, 1, 1], 0.85, 1.15)          # brightness
    return tf.clip_by_value(x, 0.0, 255.0), y


def make_dataset(x: np.ndarray, y, idx: np.ndarray, batch_size: int, training: bool = False) -> tf.data.Dataset:
    """Batches of (pixels, label) from the in-RAM uint8 arrays; y=None gives pixels only (in order)."""
    ds = tf.data.Dataset.from_tensor_slices(np.asarray(idx, dtype=np.int64))
    if training:
        ds = ds.shuffle(len(idx), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=training)

    if y is None:
        def load(i):
            return x[i]
        ds = ds.map(lambda i: tf.numpy_function(load, [i], tf.uint8), num_parallel_calls=AUTOTUNE)
        ds = ds.map(lambda b: tf.cast(tf.ensure_shape(b, [None, 224, 224, 3]), tf.float32))
    else:
        def load(i):
            return x[i], y[i].astype(np.float32).reshape(-1, 1)
        ds = ds.map(lambda i: tf.numpy_function(load, [i], [tf.uint8, tf.float32]), num_parallel_calls=AUTOTUNE)
        ds = ds.map(lambda b, l: (tf.cast(tf.ensure_shape(b, [None, 224, 224, 3]), tf.float32),
                                  tf.ensure_shape(l, [None, 1])))
        if training:
            ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


train_ds = make_dataset(train_x, train_y, tr_idx, BATCH_SIZE, training=True)
val_ds = make_dataset(train_x, train_y, va_idx, 64)

# Check one augmented batch.
xb, yb = next(iter(train_ds))
print('batch', xb.shape, xb.dtype, 'labels', yb.shape, 'pixel range', float(tf.reduce_min(xb)), float(tf.reduce_max(xb)))
fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
for ax, img, lab in zip(axes, xb.numpy()[:8], yb.numpy()[:8, 0]):
    ax.imshow(img.astype(np.uint8)); ax.set_title('person' if lab else 'non-person', fontsize=9); ax.axis('off')
plt.show()

## Training
Stage 1 trains only the head. Stage 2 unfreezes the backbone. After every epoch, the weights are saved
if validation accuracy improved, so the best epoch is kept even if a later one overfits.

In [ ]:
ckpt_path = f'{WORK_DIR}/best.weights.h5'

# Stage 1: frozen backbone, new head only.
set_backbone_trainable(backbone, False)
compile_model(model, 1e-3)
print('trainable params:', sum(int(np.prod(v.shape)) for v in model.trainable_weights))

ckpt1 = keras.callbacks.ModelCheckpoint(ckpt_path, monitor='val_acc', mode='max',
                                        save_best_only=True, save_weights_only=True, verbose=1)
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=STAGE1_EPOCHS, callbacks=[ckpt1])
best_stage1 = max(hist1.history['val_acc'])
print('stage 1 best val accuracy:', round(best_stage1, 4))

In [ ]:
# Stage 2: fine-tune the whole backbone (BatchNorm frozen), warmup then cosine decay of the learning rate.
set_backbone_trainable(backbone, True)
steps_per_epoch = len(tr_idx) // BATCH_SIZE
total_steps = steps_per_epoch * FINETUNE_EPOCHS
warmup_steps = min(500, total_steps // 10)
schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=FINETUNE_LR / 100, warmup_target=FINETUNE_LR, warmup_steps=warmup_steps,
    decay_steps=max(1, total_steps - warmup_steps), alpha=0.01)
compile_model(model, schedule)
print('trainable params:', sum(int(np.prod(v.shape)) for v in model.trainable_weights))

ckpt2 = keras.callbacks.ModelCheckpoint(ckpt_path, monitor='val_acc', mode='max', save_best_only=True,
                                        save_weights_only=True, verbose=1, initial_value_threshold=best_stage1)
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=FINETUNE_EPOCHS, callbacks=[ckpt2])

In [ ]:
# Learning curves (stage 1 then stage 2).
h = {k: hist1.history[k] + hist2.history[k] for k in ['acc', 'val_acc', 'loss', 'val_loss']}
epochs = np.arange(1, len(h['acc']) + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5))
a1.plot(epochs, h['acc'], 'o-', label='train'); a1.plot(epochs, h['val_acc'], 'o-', label='validation')
a1.set_title('accuracy @ 0.5'); a1.set_xlabel('epoch'); a1.legend()
a2.plot(epochs, h['loss'], 'o-', label='train'); a2.plot(epochs, h['val_loss'], 'o-', label='validation')
a2.set_title('loss'); a2.set_xlabel('epoch'); a2.legend()
for a in (a1, a2):
    a.axvline(STAGE1_EPOCHS + 0.5, color='gray', ls=':')
plt.tight_layout(); plt.show()
print('best val accuracy:', round(max(h['val_acc']), 4))

## Final model
The best checkpoint is loaded into a fresh **float32** copy of the same model, so the submitted model gives the
same results on any machine (GPU or CPU). Then:
* validation accuracy is re-measured with this exact model;
* if a decision threshold other than 0.5 is clearly better on validation, it is folded into the last layer's bias
  (`sigmoid(z - logit(t))` crosses 0.5 exactly where `sigmoid(z)` crosses `t`), so the graded 0.5 cutoff uses it.

In [ ]:
keras.backend.clear_session()
final_model, _, _ = build_model(load_weights=False)      # float32, same architecture
final_model.load_weights(ckpt_path)
final_model.jit_compile = False

p_val = final_model.predict(make_dataset(train_x, None, va_idx, 64), verbose=0).ravel()
y_val = train_y[va_idx]
acc_05 = ((p_val >= 0.5) == y_val).mean()
print(f'val accuracy @0.5: {acc_05:.4f}')

ts = np.round(np.arange(0.30, 0.701, 0.01), 2)
accs = np.array([((p_val >= t) == y_val).mean() for t in ts])
t_best = float(ts[accs == accs.max()][np.argmin(np.abs(ts[accs == accs.max()] - 0.5))])
print(f'best threshold on val: {t_best} -> {accs.max():.4f}')

# Fold only a clear gain (at least 0.2 points); otherwise keep 0.5.
if accs.max() - acc_05 >= 0.002:
    layer = final_model.get_layer('person_prob')
    w, b = layer.get_weights()
    layer.set_weights([w, b - np.log(t_best / (1 - t_best))])
    p_val = final_model.predict(make_dataset(train_x, None, va_idx, 64), verbose=0).ravel()
    print(f'folded threshold {t_best} into the bias; val accuracy @0.5 is now {((p_val >= 0.5) == y_val).mean():.4f}')
else:
    print('kept the 0.5 threshold')

In [ ]:
# Probabilities of the person label for the score segment, in the original order.
score_p = final_model.predict(make_dataset(score_x, None, np.arange(len(score_x)), 64), verbose=1)
score_y_hat = pd.DataFrame(score_p.astype(np.float32), columns=['score'])
assert score_y_hat['score'].dtype == np.float32
assert np.isfinite(score_y_hat['score']).all() and score_y_hat['score'].between(0, 1).all()
if LIMIT is None:
    assert score_y_hat.shape == (20000, 1)
score_y_hat.to_parquet(f'{MODEL_DIR}/score_y_hat.parquet')

with open(f'{MODEL_DIR}/model_architecture.json', 'w') as f:
    f.write(json.dumps(json.loads(final_model.to_json()), indent=True))
keras.models.save_model(final_model, f'{MODEL_DIR}/model.keras')

print('predicted person share on score:', (score_y_hat['score'] >= 0.5).mean().round(4),
      '| in train:', train_y.mean().round(4))
print('files:', sorted(os.listdir(MODEL_DIR)))

In [ ]:
# Check: the saved model file, loaded the way a grader would, reproduces the submitted scores.
reloaded = keras.models.load_model(f'{MODEL_DIR}/model.keras')
p_re = reloaded.predict(score_x[:256].astype(np.float32), batch_size=64, verbose=0).ravel()
diff = np.abs(p_re - score_y_hat['score'].values[:256]).max()
print('max abs difference vs submitted scores:', diff)
assert diff < 1e-3
check = pd.read_parquet(f'{MODEL_DIR}/score_y_hat.parquet')
print(check.shape, check.dtypes.to_dict())

# Files to upload:
# 1. this notebook
# 2. pnp_model/model_architecture.json  -- the model's definition
# 3. pnp_model/model.keras              -- the model's trained parameters
# 4. pnp_model/score_y_hat.parquet      -- person probabilities for the score segment (20K x 1, float32)